# **Main Notebook: AI School 3: CNN and Neural Style Transfer**

# **CNN with PyTorch**

#### Importing stuff

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

#### Loading the dataset

In [ ]:
# Define transformations (convert to tensor and normalize)
transform = transforms.Compose([
    transforms.ToTensor(),

    # Normalizes the pixel values to center them around zero - help the model train more effectively
    # Values go from [0,1] to [-1,1]
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load training and test datasets
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=32, shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=32, shuffle=False)

'''
Note:
batch_size: number of samples processed together before updating model weights.
shuffle: determines whether data is randomly shuffled before each epoch (helps with training, but not needed for evaluation).
trainset/testset: raw data you will train/test on.
trainloader/testloader: wrapper around the trainset/testset to allow loading in batches.
'''

#### Visualizing the data

In [ ]:
# What classes are there?
print(trainset.classes)

# Define a function to show images
def imshow(img):
    img = img / 2 + 0.5 # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

# Get a batch of training data
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Show images
imshow(torchvision.utils.make_grid(images))

#### Implementing the CNN model

Max pooling - reduces the spatial dimensions (height and width) of feature maps while retaining important information. Operates on a fixed-size sliding window (or kernel), and for each window, it outputs the maximum value within that window.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # First convolutional layer: 3 input channels, 6 output channels, 3x3 kernel size
        self.conv1 = nn.Conv2d(3, 6, 3, padding=1)
        # Second convolutional layer: 6 input channels, 16 output channels, 3x3 kernel size
        self.conv2 = nn.Conv2d(6, 16, 3, padding=1)

        # Fully connected layers
        self.fc1 = nn.Linear(16 * 8 * 8, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # Convolutional layer 1 + ReLU + Max pooling
        x = torch.relu(self.conv1(x))
        x = torch.max_pool2d(x, 2)

        # Convolutional layer 2 + ReLU + Max pooling
        x = torch.relu(self.conv2(x))
        x = torch.max_pool2d(x, 2)

        # Flatten the tensor for the fully connected layer
        x = x.view(-1, 16 * 8 * 8)

        # Fully connected layers + ReLU
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))

        # Final output layer
        x = self.fc3(x)
        return x

#### Training the model

We are using Adam which is a more sophisticated optimizer than plain gradient descent as it usually performs better. More on Adam [here](https://www.geeksforgeeks.org/adam-optimizer/#).

In [ ]:
# Define the device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN().to(device)

# Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
epochs = 10
for epoch in range(epochs):
    running_loss = 0.0 # total loss for current epoch
    for inputs, labels in trainloader:
        # Ensure that data is on the same device as the model
        inputs, labels = inputs.to(device), labels.to(device)

        # Zeros out the gradients accumulated from the previous step
        optimizer.zero_grad()

        # Forward + backward + optimize
        outputs = model(inputs) # predictions
        loss = criterion(outputs, labels) # loss calculation
        loss.backward() # backprop - computes gradients of the loss
        optimizer.step() # updates model's parameters

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader)}")

#### Individual prediction

Try changing img_idx to see different images and corresponding predicted label. You will see that sometimes the model got it right, and sometimes it didn't

In [ ]:
dataiter = iter(testloader)
images, labels = next(dataiter)

img_idx = 9 # Change this is you'd like
image = images[img_idx]
label = labels[img_idx]

# Visualize the chosen image
print("Actual Label:", trainset.classes[label])
imshow(image)

# Prepare the image for prediction by adding a batch dimension and moving to the model's device
image = image.to(device).unsqueeze(0)

# Make a prediction
model.eval()  # Set model to evaluation mode
with torch.no_grad(): # Don't calculate gradients
    outputs = model(image) # Predict
    _, predicted = torch.max(outputs, 1) # Choose class with max predicted probability
    predicted_label = trainset.classes[predicted.item()]

print("Predicted Label:", predicted_label)

#### Evaluating the model

In [ ]:
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Accuracy on test set: {100 * correct / total:.2f}%")

Try adjusting this code to increase accuracy. Be careful - more complexity in the model means longer runtime - we want to be able to train the model in reasonable amount of time. Share your result with others if you were able to beat this implementation! Changes to consider:

1. Model Architecture

Add more layers: Use model with additional convolutional and pooling layers. This can help the model capture more complex patterns.

Increase filter sizes and count: Adding more filters in each convolutional layer (e.g., going from 32 filters to 64 or 128) can improve feature extraction.

2. Data Augmentation

Apply random crops and horizontal flips: This can help the model generalize by seeing slightly modified versions of each image.

Color jitter: Randomly adjust brightness, contrast, and saturation to make the model robust to variations in color.

3. Optimization and Regularization Techniques

Try different optimizers: While Adam is generally a good choice, trying alternatives like SGD can sometimes yield better results.

Dropout: Adding [dropout layers](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html) can help prevent overfitting by forcing the network to learn more robust features.

4. Hyperparameter Tuning

Learning rate: Experiment with different learning rates (e.g., starting from 0.001, 0.01, or even 0.1).

Batch size: Test different batch sizes (e.g., 32, 64, or 128). Smaller batch sizes often help with generalization, while larger batches can speed up training.

Increase number of epochs: Training for more epochs can give the model more time to learn, but this also increases the risk of overfitting.

5. There are a lot of other techniques that you can explore and try

# **Workshop Project: Implementing Neural Style Transfer!**

In [ ]:
# importing libraries to implement style-transfer

from __future__ import print_function

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from PIL import Image
import matplotlib.pyplot as plt

import torchvision.transforms as transforms
import torchvision.models as models

import copy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# STEP 1: Load content and style images with proper size
# Set desired size of output image - smaller if using CPU
imsize = 512 if torch.cuda.is_available() else 128

loader = transforms.Compose([
    transforms.Resize((imsize, imsize)),  # Scale image to desired size
    transforms.ToTensor()                 # Convert to tensor
])

# Helper function to load image and format it
def image_loader(image_name):
    image = Image.open(image_name)
    image = loader(image).unsqueeze(0)  # Add fake batch dimension for compatibility
    return image.to(device, torch.float)

# Upload your chosen style and content images beforehand
style_file = "cubism.jpg"
content_file = "President_Barack_Obama.jpg"
style_img = image_loader(style_file)
content_img = image_loader(content_file)

assert style_img.size() == content_img.size(), \
    "Style and content images must be the same size"

# Define function to display images for visualization
unloader = transforms.ToPILImage()

def imshow(tensor, title=None):
    image = tensor.cpu().clone()
    image = image.squeeze(0)
    image = unloader(image)
    plt.imshow(image)
    if title is not None:
        plt.title(title)
    plt.pause(0.001)

# Display content and style images
plt.figure()
imshow(style_img, title='Style Image')
plt.figure()
imshow(content_img, title='Content Image')


**Defining content and style loss**

In [ ]:
class ContentLoss(nn.Module):
    def __init__(self, target):
        super(ContentLoss, self).__init__()
        # TODO: Detach target to avoid computing gradients

    def forward(self, input):
        # TODO: MSE loss between input and target features


In [ ]:
def gram_matrix(input):
    a, b, c, d = input.size()  # a=batch size(=1)
    # b=number of feature maps
    # (c,d)=dimensions of a f. map (N=c*d)

    features = input.view(a * b, c * d)  # resise F_XL into \hat F_XL

    G = torch.mm(features, features.t())  # compute the gram product

    # we 'normalize' the values of the gram matrix
    # by dividing by the number of element in each feature maps.
    return G.div(a * b * c * d)

In [ ]:
class StyleLoss(nn.Module):
# TODO: Implement Style Loss
    def __init__(self, target_feature):

    def forward(self, input):


In [ ]:
# TODO: Import pretrained model

In [ ]:
cnn_normalization_mean = torch.tensor([0.485, 0.456, 0.406]).to(device)
cnn_normalization_std = torch.tensor([0.229, 0.224, 0.225]).to(device)

# create a module to normalize input image so we can easily put it in a
# nn.Sequential
class Normalization(nn.Module):
    def __init__(self, mean, std):
        super(Normalization, self).__init__()
        # .view the mean and std to make them [C x 1 x 1] so that they can
        # directly work with image Tensor of shape [B x C x H x W].
        # B is batch size. C is number of channels. H is height and W is width.
        self.mean = torch.tensor(mean).view(-1, 1, 1)
        self.std = torch.tensor(std).view(-1, 1, 1)

    def forward(self, img):
        # normalize img
        return (img - self.mean) / self.std

In [ ]:
# desired depth layers to compute style/content losses :
content_layers_default = ['conv_4']
style_layers_default = ['conv_1', 'conv_2', 'conv_3', 'conv_4', 'conv_5']

def get_style_model_and_losses(cnn, normalization_mean, normalization_std,
                               style_img, content_img,
                               content_layers=content_layers_default,
                               style_layers=style_layers_default):
    cnn = copy.deepcopy(cnn)

    # normalization module
    normalization = Normalization(normalization_mean, normalization_std).to(device)

    # just in order to have an iterable access to or list of content/syle
    # losses
    content_losses = []
    style_losses = []

    # assuming that cnn is a nn.Sequential, so we make a new nn.Sequential
    # to put in modules that are supposed to be activated sequentially
    model = nn.Sequential(normalization)

    i = 0  # increment every time we see a conv
    for layer in cnn.children():
        if isinstance(layer, nn.Conv2d):
            i += 1
            name = 'conv_{}'.format(i)
        elif isinstance(layer, nn.ReLU):
            name = 'relu_{}'.format(i)
            # The in-place version doesn't play very nicely with the ContentLoss
            # and StyleLoss we insert below. So we replace with out-of-place
            # ones here.
            layer = nn.ReLU(inplace=False)
        elif isinstance(layer, nn.MaxPool2d):
            name = 'pool_{}'.format(i)
        elif isinstance(layer, nn.BatchNorm2d):
            name = 'bn_{}'.format(i)
        else:
            raise RuntimeError('Unrecognized layer: {}'.format(layer.__class__.__name__))

        model.add_module(name, layer)

        if name in content_layers:
            # add content loss:
            target = model(content_img).detach()
            content_loss = ContentLoss(target)
            model.add_module("content_loss_{}".format(i), content_loss)
            content_losses.append(content_loss)

        if name in style_layers:
            # add style loss:
            target_feature = model(style_img).detach()
            style_loss = StyleLoss(target_feature)
            model.add_module("style_loss_{}".format(i), style_loss)
            style_losses.append(style_loss)

    # now we trim off the layers after the last content and style losses
    for i in range(len(model) - 1, -1, -1):
        if isinstance(model[i], ContentLoss) or isinstance(model[i], StyleLoss):
            break

    model = model[:(i + 1)]

    return model, style_losses, content_losses

In [ ]:
input_img = content_img.clone()
# if you want to use white noise instead uncomment the below line:
# input_img = torch.randn(content_img.data.size(), device=device)

# add the original input image to the figure:
plt.figure()
imshow(input_img, title='Input Image')

In [ ]:
def get_input_optimizer(input_img):
    # this line to show that input is a parameter that requires a gradient
    # TODO: Make image learnable and pass to optimizer:
    optimizer = optim.Adam([input_img.requires_grad_()], lr=1e-2)
    return optimizer

In [ ]:
def run_style_transfer(cnn, normalization_mean, normalization_std,
                       content_img, style_img, input_img, num_steps=300,
                       style_weight=1e6, content_weight=1e-2):
    """Run the style transfer."""
    print('Building the style transfer model..')
    model, style_losses, content_losses = get_style_model_and_losses(cnn,
        normalization_mean, normalization_std, style_img, content_img)
    optimizer = get_input_optimizer(input_img)
    print(model)
    print('Optimizing..')
    run = [0]
    while run[0] <= num_steps:

        # correct the values of updated input image
        input_img.data.clamp_(0, 1)

        # Wipe .grad of all learning tensors
        optimizer.zero_grad()

        # TODO: Perform the forward pass with the input image:
        model(input_img)


        # TODO: Tally up the style and content losses.
        # Loop over loss modules and accumulate the stored per-layer losses.


        # TODO: Weight style and content


        # TODO: What is the overall loss?


        loss.backward()

        run[0] += 1
        if run[0] % 50 == 0:
            print("run {}:".format(run))
            print('Style Loss : {:4f} Content Loss: {:4f}'.format(
                style_score.item(), content_score.item()))
            print()
            plt.figure()
            plt.xticks([])
            plt.yticks([])
            imshow(input_img, title=f'Styled Output at iteration #{run[0]}')
            plt.show()

        optimizer.step()

    # a last correction...
    input_img.data.clamp_(0, 1)

    return input_img

In [ ]:
styled_output = run_style_transfer(cnn, cnn_normalization_mean, cnn_normalization_std,
                            content_img, style_img, input_img, num_steps=1500)

plt.figure()
plt.xticks([])
plt.yticks([])
imshow(styled_output, title='Styled Output')

plt.show()

In [ ]:
plt.figure()
plt.xticks([])
plt.yticks([])
imshow(styled_output, title='Styled Output')

plt.show()

In [ ]:
# save styled image
from torchvision.utils import save_image
save_image(styled_output, "obama_styled.jpg")